Week04_Fri

Link Github: https://github.com/BM910/AI-ARIN330585

Ma trận 2 chiều "room" biểu diễn sàn nhà:
    0 - Không có bụi, có thể đi qua được.
    1 - Có bụi, có thể đi qua được.
    2 - Vật cản, không đi qua được.
    3 - Máy hút bụi.

In [1]:
from collections import deque
import copy
import tkinter as tk
import random

current_room = None

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0, parent_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.cost = self.calculate_dirty_value() + parent_cost

    def __str__(self):
        state = ""
        for row in self.state:
            state += str(row) + '\n'
        move_and_cost = f"Moved {self.action}\tStep {self.path_cost}\tCost: {self.cost}"
        return state + move_and_cost

    def calculate_dirty_value(self):
        total_dirty_value = 0
        dirty_tile_value = 2
        for i in range(len(self.state)):
            for j in range(len(self.state[0])):
                if self.state[i][j] == 1:
                    total_dirty_value += dirty_tile_value
        return total_dirty_value
    
    def is_clean(self):
        for row in self.state:
            for tile in row:
                if tile == 1:
                    return False
        return True

    def find_roomba(self):
        for i in range(len(self.state)):
            for j in range(len(self.state[0])):
                if self.state[i][j] == 3:
                    return (i, j)

    def get_moves(self):
        full_moves = [(-1, 0, "UP"), (1, 0, "DOWN"), (0, -1, "LEFT"), (0, 1, "RIGHT")]
        valid_moves = []
        for dx, dy, action in full_moves:
            if self.check_valid_move(dx, dy):
                x, y = self.find_roomba()
                nx = x + dx
                ny = y + dy
                if self.state[nx][ny] == 1:
                    action += f" and cleaned [{nx}][{ny}]"
                valid_moves.append((dx, dy, action))
        return valid_moves
                    
                

    def check_valid_move(self, dx, dy):
        x, y = self.find_roomba()
        nx = x + dx
        ny = y + dy
        if 0 <= nx < len(self.state) and 0 <= ny < len(self.state[0]) and self.state[nx][ny] != 2:
            return True
        
    def execute_move(self, dx, dy, return_state=False):
        x, y = self.find_roomba()
        nx = x + dx
        ny = y + dy
        if self.check_valid_move(dx, dy):
            if return_state:
                next_state = copy.deepcopy(self.state)
                next_state[x][y] = 0
                next_state[nx][ny] = 3
                return next_state
            else:
                self.state[x][y] = 0
                self.state[nx][ny] = 3
            return True
        return False
    
    def get_parent_list(self):
        path = [self]
        while self.parent:
            path.append(self.parent)
            self = self.parent
        return path[::-1]

def bfs1(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)
    if node.is_clean():
        return node

    frontier = deque()
    reached = set()
    frontier.append(node)
    max_frontier = 1
    frontier_compare_count = 0

    while frontier:
        node = frontier.popleft()
        reached.add(str(node.state))

        if node.is_clean():
                update_log("BFS_1")
                update_log(f"frontier[] max size: {max_frontier}")
                update_log(f"reached() size: {len(reached)}")
                update_log("-"*20)
                return node
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.execute_move(dx, dy, return_state=True), node, action)
            child_is_unique = True

            for frontier_node in frontier:
                frontier_compare_count += 1
                if child_node.state == frontier_node.state:
                    child_is_unique = False
                    break

            if child_is_unique and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def bfs2(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)
    if node.is_clean():
        return node

    frontier = deque()
    reached = set()
    frontier.append(node)
    max_frontier = 1

    while frontier:
        node = frontier.popleft()
        reached.add(str(node.state))
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.execute_move(dx, dy, return_state=True), node, action)
            if child_node.is_clean():
                    update_log("BFS_2")
                    update_log(f"frontier[] max size: {max_frontier}")
                    update_log(f"reached() size: {len(reached)}")
                    update_log("-"*20)
                    return child_node
            child_is_unique = True

            for frontier_node in frontier:
                if child_node.state == frontier_node.state:
                    child_is_unique = False
                    break

            if child_is_unique and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def dfs1(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)
    if node.is_clean():
        return node

    frontier = deque()
    reached = set()
    frontier.append(node)
    max_frontier = 1

    while frontier:
        node = frontier.pop()
        reached.add(str(node.state))

        if node.is_clean():
                update_log("DFS_1")
                update_log(f"frontier[] max size: {max_frontier}")
                update_log(f"reached() size: {len(reached)}")
                update_log("-"*20)
                return node
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.execute_move(dx, dy, return_state=True), node, action)
            child_is_unique = True

            for frontier_node in frontier:
                if child_node.state == frontier_node.state:
                    child_is_unique = False
                    break

            if child_is_unique and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def dfs2(initial_state):
    if not initial_state:
        return
    node = Node(initial_state)
    if node.is_clean():
        return node

    frontier = deque()
    reached = set()
    frontier.append(node)
    max_frontier = 1

    while frontier:
        node = frontier.pop()
        reached.add(str(node.state))
        
        moves = node.get_moves()
        
        for dx, dy, action in moves:
            child_node = Node(node.execute_move(dx, dy, return_state=True), node, action)
            if child_node.is_clean():
                    update_log("DFS_2")
                    update_log(f"frontier[] max size: {max_frontier}")
                    update_log(f"reached() size: {len(reached)}")
                    update_log("-"*20)
                    return child_node
            child_is_unique = True

            for frontier_node in frontier:
                if child_node.state == frontier_node.state:
                    child_is_unique = False
                    break

            if child_is_unique and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def ids(initial_state, max_depth=1000):
    if not initial_state:
        return
    node = Node(initial_state)
    if node.is_clean():
        return node
    
    for depth in range(max_depth):
        result = dls(initial_state, depth)
        if result != "cutoff":
            return result

def dls(initial_state, limit):
    frontier = [Node(initial_state)]
    reached = set()
    result = None
    max_frontier = 1

    while frontier:
        node = frontier.pop()

        if node.is_clean():
            update_log("IDS")
            update_log(f"frontier[] max size: {max_frontier}")
            update_log(f"reached() size: {len(reached)}")
            update_log("-"*20)
            return node
        
        if node.path_cost >= limit:
            result = "cutoff"
            continue
        
        is_cycle = False
        for frontier_node in frontier:
            if frontier_node.state == node.state:
                is_cycle = True
                break

        if not is_cycle and str(node.state) not in reached:
            moves = node.get_moves()
            for dx, dy, action in moves:
                child_node = Node(node.execute_move(dx, dy, return_state=True), node, action, path_cost=node.path_cost + 1, parent_cost=node.cost)
                frontier.append(child_node)
            reached.add(str(node.state))
        max_frontier = max(max_frontier, len(frontier))

    return result

def ucs(initial_state):
    frontier = [Node(initial_state)]
    reached = set()
    max_frontier = 1

    while frontier:
        idx = 0
        min_cost = frontier[0].cost
        for i in range(1, len(frontier)):
            if min_cost > frontier[i].cost:
                idx = i
                min_cost = frontier[i].cost

        node = frontier.pop(idx)
        if node.is_clean():
            update_log("UCS")
            update_log(f"frontier[] max size: {max_frontier}")
            update_log(f"reached() size: {len(reached)}")
            update_log("-"*20)
            return node

        state_str = str(node.state)
        if state_str in reached:
            continue
        reached.add(state_str)

        moves = node.get_moves()
        for move in moves:
            dx, dy, action = move
            if node.check_valid_move(dx, dy):
                next_state = copy.deepcopy(node.state)

                temp_node = Node(next_state)
                temp_node.execute_move(dx, dy)

                if str(temp_node.state) not in reached:
                    child_node = Node(temp_node.state, node, action, path_cost=node.path_cost + 1, parent_cost=node.cost)
                    frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def randomize_map():
    global current_room

    rows = random.randint(3, 7)
    cols = rows

    choices = [0, 1, 2]
    weights = [0.60, 0.40, 0.20]

    current_room = []
    for r in range(rows):
        row_tiles = random.choices(choices, weights=weights, k=cols)
        current_room.append(row_tiles)
        
    roomba_placed = False
    while not roomba_placed:
        random_row = random.randint(0, rows - 1)
        random_col = random.randint(0, cols - 1)
        
        if current_room[random_row][random_col] in (0, 1):
            current_room[random_row][random_col] = 3
            roomba_placed = True

    draw_matrix_on_map(current_room)

def draw_matrix_on_map(matrix):
    # Clear anything previously drawn on the map canvas
    map_canvas.delete("all")
    
    rows = len(matrix)
    cols = len(matrix[0])
    
    # Dynamically scale cell size to perfectly fit your 400x400 map area
    cell_width = 400 // cols
    cell_height = 400 // rows
    
    # Map out the color scheme per rule
    colors = {
        0: "lightgray",  # Clean, passable floor
        1: "#7A592F",    # Dirty floor
        2: "#1B1A1A",    # Obstacle
        3: "green"     # Roomba
    }
    
    for r in range(rows):
        for c in range(cols):
            val = matrix[r][c]
            color = colors.get(val, "white")
            
            # Calculate pixel bounds for each grid coordinate
            x1 = c * cell_width
            y1 = r * cell_height
            x2 = x1 + cell_width
            y2 = y1 + cell_height
            
            # Draw the square onto the canvas
            map_canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="black", width=2)
            
            # Optional visual aid: Draw a distinct circle if it's the Roomba
            if val == 3:
                map_canvas.create_oval(x1+5, y1+5, x2-5, y2-5, fill="gray", outline="white", width=2)

def run_and_animate_algo(algo_func):
    global current_room
    if current_room is None:
        update_log("Please generate a map first using Randomize!")
        return

    final_node = algo_func(copy.deepcopy(current_room))
    
    if final_node is None:
        update_log("No solution found to clean the room!")
        return
        
    # path_timeline: [(state_1, action_1), (state_2, action_2), ...]
    parent_list = final_node.get_parent_list()
    path_timeline = []
    for node in parent_list:
        path_timeline.append((node.state, str(node.action)))
    
    animate_steps(path_timeline, 0)

def animate_steps(path_timeline, step_index):
    if step_index >= len(path_timeline):
        update_log("Animation complete! Room is clean.\n")
        return

    room_state, action_text = path_timeline[step_index]
    
    draw_matrix_on_map(room_state)
    
    update_log("Roomba moved " + action_text)
    update_log("-"*20)

    window.after(50, lambda: animate_steps(path_timeline, step_index + 1))

def log_path_timeline(path_timeline):
    moves = ""
    for state, step in path_timeline:
        step = step.split(maxsplit=1)[0]
        moves += step + " -> "
    moves += "DONE\n"
    update_log(f"Path: {moves}")

def update_log(message="", clear=False):
    if clear:
        log_text.delete("1.0", tk.END)
    if message != "":
        log_text.insert(tk.END, message + "\n")
    
    log_text.see(tk.END)

# --- WINDOW SETUP ---
window = tk.Tk()
window.title("Layout Fix")

window.columnconfigure(0, weight=15)
window.columnconfigure(1, weight=40)
window.columnconfigure(2, weight=25)
window.rowconfigure(0, weight=1)
window.rowconfigure(1, weight=0)

# Sidebar Frame
frm_algo_options = tk.Frame(bg="gray", width=150, height=400, relief=tk.SUNKEN, border=5)
frm_algo_options.grid(column=0, row=0, sticky="nsew")
frm_algo_options.grid_propagate(False)
frm_algo_options.columnconfigure(0, weight=1)

# Map Frame (Replacing generic frame with a Canvas widget for pixel manipulation)
map_canvas = tk.Canvas(master=window, bg="gray", width=400, height=400, relief=tk.SUNKEN, border=5)
map_canvas.grid(column=1, row=0, sticky="nsew")
map_canvas.grid_propagate(False)

# Log Frame
frm_state_log = tk.Frame(bg="gray", width=250, height=400, relief=tk.SUNKEN, border=5)
frm_state_log.grid(column=2, row=0, sticky="nsew")
frm_state_log.grid_propagate(False)

# Actions Frame
frm_actions = tk.Frame(bg="darkgray", width=800, height=50, relief=tk.SUNKEN, border=5)
frm_actions.grid(column=0, columnspan=3, row=1, sticky="nsew")
frm_actions.grid_propagate(False)


# --- BUTTONS ---
btn_font = ("Helvetica", 12, "bold")

btn_bfs1 = tk.Button(frm_algo_options, text="BFS_1", bg="white", font=btn_font, command=lambda: run_and_animate_algo(bfs1))
btn_bfs1.grid(column=0, row=0, sticky="ew", padx=10, pady=5)

btn_bfs2 = tk.Button(frm_algo_options, text="BFS_2", bg="white", font=btn_font, command=lambda: run_and_animate_algo(bfs2))
btn_bfs2.grid(column=0, row=1, sticky="ew", padx=10, pady=5)

btn_dfs1 = tk.Button(frm_algo_options, text="DFS_1", bg="white", font=btn_font, command=lambda: run_and_animate_algo(dfs1))
btn_dfs1.grid(column=0, row=2, sticky="ew", padx=10, pady=5)

btn_dfs2 = tk.Button(frm_algo_options, text="DFS_2", bg="white", font=btn_font, command=lambda: run_and_animate_algo(dfs2))
btn_dfs2.grid(column=0, row=3, sticky="ew", padx=10, pady=5)

btn_ids = tk.Button(frm_algo_options, text="IDS", bg="white", font=btn_font, command=lambda: run_and_animate_algo(ids))
btn_ids.grid(column=0, row=4, sticky="ew", padx=10, pady=5)

btn_ucs = tk.Button(frm_algo_options, text="UCS", bg="white", font=btn_font, command=lambda: run_and_animate_algo(ucs))
btn_ucs.grid(column=0, row=5, sticky="ew", padx=10, pady=5)

# Algorithm buttons and Randomize button spacing
frm_algo_options.rowconfigure(6, weight=1) 

btn_random = tk.Button(frm_algo_options, text="Randomize", bg="white", font=btn_font, command=randomize_map)
btn_random.grid(column=0, row=7, sticky="ew", padx=10, pady=10)

btn_clr_log = tk.Button(frm_algo_options, text="Clear log", bg="white", font=btn_font, command=lambda: update_log(clear=True))
btn_clr_log.grid(column=0, row=8, sticky="ew", padx=10, pady=10)

# --- LOG WIDGET SETUP ---
log_text = tk.Text(frm_state_log, bg="white", wrap=tk.WORD, font=("Consolas", 10))
log_scroll = tk.Scrollbar(frm_state_log, command=log_text.yview)
log_text.configure(yscrollcommand=log_scroll.set)

log_scroll.pack(side=tk.RIGHT, fill=tk.Y)
log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

window.mainloop()